# Track B RL training on Kaggle GPU

Trains the LearnedScorer on `best_deck.csv` with the fixed reward, distills, SPRT-gates, and packages.

## Before running (notebook settings, right sidebar)
1. **Accelerator -> GPU** (T4 x2 or P100).
2. **Internet -> On** (needed for `git clone` + `pip install`).
3. **Add Input -> Competitions -> `pokemon-tcg-ai-battle`** (ships the Linux engine `libcg.so`).

Then Run All. Outputs land in `/kaggle/working/` for download. No Kaggle submission is made.

In [ ]:
# 1. Get the code (latest main). If the repo is PRIVATE, set a GitHub PAT as a
#    Kaggle Secret named GITHUB_PAT and use the commented clone line instead.
import os, shutil
REPO = 'https://github.com/TomBombadyl/kaggle_pokemon.git'
if os.path.isdir('/kaggle/working/kaggle_pokemon'):
    shutil.rmtree('/kaggle/working/kaggle_pokemon')
%cd /kaggle/working
!git clone --depth 1 {REPO}
# --- private repo alternative ---
# from kaggle_secrets import UserSecretsClient
# pat = UserSecretsClient().get_secret('GITHUB_PAT')
# !git clone --depth 1 https://{pat}@github.com/TomBombadyl/kaggle_pokemon.git
%cd /kaggle/working/kaggle_pokemon
!git log --oneline -1

In [ ]:
# 2. Training-only deps (torch is preinstalled on Kaggle GPU images).
!pip install -q 'gymnasium>=0.29' 'stable-baselines3>=2.3' 'sb3-contrib>=2.3'
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# 3. Wire the cabt engine from the attached competition into the path our code
#    expects (data/sim/sample_submission/cg). Verifies import + battle_start.
!python scripts/kaggle_setup.py

In [ ]:
# 4. Train -> distill -> SPRT gate @40g -> package. n-envs 4 = Kaggle CPU cores.
#    Kyogre is held out of training as a generalization probe. ~10-20 min.
#    If the eval callback ever crashes on this image, add --no-eval.
!python scripts/train_track_b_deck.py \
    --deck report/rl_deck_campaign/best_deck.csv --slug rl_deck \
    --timesteps 100000 --n-envs 4 --opponents benchmark --holdout a2_kyogre \
    --gate-games 40 --package --promote

In [ ]:
# 5. Collect artifacts into /kaggle/working for download (Output tab).
import glob, shutil, os
os.makedirs('/kaggle/working/out', exist_ok=True)
for p in (glob.glob('dist/candidates/track_b_learned_rl_deck.tar.gz')
          + glob.glob('agent/models/distilled_rl_deck_v1.npz')
          + glob.glob('agent/models/rl_policy.zip')
          + glob.glob('report/track_b_gates/*rl_deck*gate.md')
          + glob.glob('report/rl_train/eval_*.json')):
    shutil.copy2(p, '/kaggle/working/out/'); print('saved', p)
print('\n--- gate report ---')
print(open(glob.glob('report/track_b_gates/*rl_deck*gate.md')[0]).read())